# CS781 Phase 2 - a1_adapters (updated)

**Architecture:** Phase 1 A1 (same original model class and Phase 1 checkpoint)  
**Adaptation:** adapters  

- 256-token windows, stride 192; original document offsets kept unchanged.
- Combine overlapping window predictions before decoding BIO tags.
- Compare original, stitched greedy and constrained Viterbi decoding on validation.
- Train up to 30 epochs; stop if strict Macro-F1 does not improve for 5 epochs.
- Save the best checkpoint in a separate output folder.

Only official fold1 and fold2 are used for training/validation. Fold3 is used for prediction only.


**Phase 1 checkpoint pinned:** `P-261110044/cs781-a1-checkpoints/checkpoint-6104/model.safetensors`. The loader checks the checkpoint keys, hidden size and layer counts, and uses the matching tokenizer.


In [1]:
# Libraries needed for the Phase 2 notebooks.
# If torchao is already imported, restart Kaggle after running this cell.
%pip -q install "transformers>=4.45,<5" huggingface_hub safetensors tokenizers peft tqdm

# Kaggle may have torchao 0.10.0, which crashes with newer PEFT.
# Standard LoRA here does not need torchao.
%pip -q uninstall -y torchao


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:

# Phase 2 - a1_adapters
# Edit HF_REPO_NAME below when your Phase 1 repo has a different name.
import os, json, random, math, time, zipfile, re
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizerFast
from huggingface_hub import HfApi, hf_hub_download

SEED = 44
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ARCH = 'a1'
TECHNIQUE = 'adapters'
EXP_ID = 'a1_adapters_v2_30ep'  # Alag folder: purana run overwrite nahi hoga
# edit ONLY the name after your username, not your username itself
HF_REPO_NAME = 'P-261110044/cs781-a1-checkpoints'  # Exact Phase 1 repo, owner included
PREFERRED_PHASE1_WEIGHT = 'checkpoint-6104/model.safetensors'
# Agar Phase 1 ka folder Kaggle Input mein hai toh uska path yahan de sakte hain.
# Example: /kaggle/input/my-phase1-artifact/final_model
LOCAL_PHASE1_DIR = ''  # Optional; HF repo ko use karna ho toh empty rakho.
# if tokenizer.json is not in the model repo, put a tokenizer repo name here
TOKENIZER_REPO_NAME = ''     # CHANGE THIS only if the checkpoint repo has no tokenizer
# alternatively, a Kaggle input directory that contains tokenizer.json
LOCAL_TOKENIZER_DIR = ''    # CHANGE THIS only if needed


# Attention-head count is not stored in PyTorch weight tensors.
# Normally the saved config / matching Phase 1 notebook is used.
# Only fill these if your Phase 1 notebook or training log confirms them.
A1_ATTENTION_HEADS = None
A2_ATTENTION_HEADS = None

OUTPUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUTPUT = OUTPUT / ('phase2_' + EXP_ID)
OUTPUT.mkdir(parents=True,exist_ok=True)

DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
MAX_LEN = 256  # Window ke andar 256 tokens ka context
STRIDE = 192  # 64-token overlap for boundary entities
EPOCHS = 30      # Maximum epochs; early stopping may finish sooner
PATIENCE = 5     # Agar 5 epochs tak val F1 improve na ho, stop
LR_ADAPT = 0.0001  # adapters method ke trainable weights ka LR
LR_HEAD = 2e-4  # Naya classification head alag LR use karega
BATCH_SIZE = 1    # avoids padding. It also works with architecture 2's encoder.
ACCUM_STEPS = 4   # update weights every 4 pieces
O_WEIGHT = 0.5   # 0.2 pe kaafi extra entities predict ho rahi thi
VAL_PART = 0.10
MAX_TRAIN_DOCS = None  # Sab available training documents
MAX_VAL_DOCS = None  # Puri validation split, sirf 5 docs nahi
DECODE_METHOD = 'viterbi'  # Used during training validation
AUTO_SELECT_DECODER = True  # At the end, choose decoding by held-out F1
RANK = 8
ALPHA = 16
# Same secret name used in your Phase 1 notebooks: HF
HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF') or HF_TOKEN
except Exception:
    pass
api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()['name']      # read the active HF account, no hard-coded username
if not HF_REPO_NAME and not LOCAL_PHASE1_DIR:
    raise ValueError('Apne Phase 1 model ka HF_REPO_NAME ya LOCAL_PHASE1_DIR set karo.')
HF_REPO = HF_REPO_NAME  # Do not replace the checkpoint owner with the login name
print('Logged in as:', HF_USER)
print('Loading Phase 1 from:', HF_REPO)
print('Experiment:', EXP_ID, TECHNIQUE)


Logged in as: P-261110044
Loading Phase 1 from: P-261110044/cs781-a1-checkpoints
Experiment: a1_adapters_v2_30ep adapters


In [3]:

# These are the ORIGINAL Phase 1 PyTorch model classes.
# Their internal transformer layers are not rewritten for Phase 2.
import torch

import torch.nn as nn

import torch.nn.functional as F

from transformers.modeling_outputs import CausalLMOutput

class DecoderBlock(nn.Module):
    """
    One Transformer decoder block.

    Structure:
        LayerNorm
        -> Causal Multi-Head Self Attention
        -> Residual Connection
        -> LayerNorm
        -> Feed Forward Network
        -> Residual Connection
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        # Normalize before self-attention.
        self.norm1 = nn.LayerNorm(
            d_model
        )

        # Multi-head self-attention.
        #
        # Query, Key and Value all come from
        # the same token sequence.
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Normalize before feed-forward network.
        self.norm2 = nn.LayerNorm(
            d_model
        )

        # Transformer feed-forward network.
        self.feed_forward = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        causal_mask
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        z = self.norm1(x)

        attention_output, _ = self.attention(
            z,
            z,
            z,
            attn_mask=causal_mask,
            need_weights=False
        )

        # Residual connection.
        x = x + self.dropout(
            attention_output
        )

        # ----------------------------------------------------
        # 2. Feed-forward network
        # ----------------------------------------------------

        z = self.norm2(x)

        ff_output = self.feed_forward(
            z
        )

        # Second residual connection.
        x = x + self.dropout(
            ff_output
        )

        return x

class DecoderTransformerLM(nn.Module):
    """
    Complete custom decoder-only Transformer
    language model.
    """

    def __init__(
        self,
        vocab_size,
        context_length=2048,
        d_model=640,
        n_heads=10,
        n_layers=8,
        d_ff=2560,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = (
            context_length
        )

        self.vocab_size = (
            vocab_size
        )

        # Safety check:
        # embedding dimension must divide evenly
        # across attention heads.
        assert (
            d_model % n_heads == 0
        ), (
            "d_model must be divisible "
            "by n_heads"
        )

        # ----------------------------------------------------
        # Token embeddings
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # ----------------------------------------------------
        # Learned positional embeddings
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            context_length,
            d_model
        )

        # ----------------------------------------------------
        # Stack of custom Transformer decoder blocks
        # ----------------------------------------------------

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization.
        self.final_norm = nn.LayerNorm(
            d_model
        )

        # Convert hidden representation
        # into vocabulary logits.
        self.output_layer = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )


    def forward(
        self,
        input_ids,
        labels=None,
        attention_mask=None,
        **kwargs
    ):

        _, seq_len = input_ids.shape

        if seq_len > self.context_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"context length {self.context_length}"
            )

        # ----------------------------------------------------
        # 1. Token embeddings
        # ----------------------------------------------------

        token_vectors = (
            self.token_embedding(
                input_ids
            )
        )

        # ----------------------------------------------------
        # 2. Positional embeddings
        # ----------------------------------------------------

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        position_vectors = (
            self.position_embedding(
                positions
            )
        )

        x = (
            token_vectors
            +
            position_vectors
        )

        # ----------------------------------------------------
        # 3. Causal mask
        #
        # True = attention blocked.
        #
        # Token i cannot see tokens after i.
        # ----------------------------------------------------

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # ----------------------------------------------------
        # 4. Transformer blocks
        # ----------------------------------------------------

        for block in self.blocks:

            x = block(
                x,
                causal_mask
            )

        # ----------------------------------------------------
        # 5. Vocabulary prediction
        # ----------------------------------------------------

        x = self.final_norm(
            x
        )

        logits = self.output_layer(
            x
        )

        loss = None

        # ----------------------------------------------------
        # 6. Next-token language modelling loss
        # ----------------------------------------------------

        if labels is not None:

            # Example:
            #
            # logits:
            # t0 t1 t2
            #
            # targets:
            # t1 t2 t3

            shift_logits = (
                logits[:, :-1, :]
                .contiguous()
            )

            shift_labels = (
                labels[:, 1:]
                .contiguous()
            )

            loss = F.cross_entropy(
                shift_logits.view(
                    -1,
                    shift_logits.size(-1)
                ),
                shift_labels.view(-1)
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits
        )

# Read the saved Phase 1 model and tokenizer from my Hugging Face repo.
# It may contain final_model/ or checkpoint-6104/ (or some other checkpoint).
import shutil
# Local Phase 1 folder use karne se A2 ke liye HF upload zaroori nahi.
# Agar folder nahi diya toh same HF repo se checkpoint download hoga.
source_dir = Path(LOCAL_PHASE1_DIR) if LOCAL_PHASE1_DIR else None
if source_dir is not None:
    if not source_dir.is_dir():
        raise FileNotFoundError('Phase 1 local folder nahi mila: ' + str(source_dir))
    repo_files = [str(q.relative_to(source_dir)) for q in source_dir.rglob('*') if q.is_file()]
else:
    repo_files = api.list_repo_files(HF_REPO, repo_type='model')
print('Phase 1 source:', source_dir if source_dir else HF_REPO)
print('Available files:', repo_files[:12])

weight_names = ('model_state.pt', 'model.safetensors', 'pytorch_model.bin')
final_folders = ('architecture-3-final/', 'architecture-4-final/',
                 'final_model/', 'legal-gpt2-final/', 'custom-decoder-final/', '')
checkpoint_folders = sorted(
    {f.split('/')[0] + '/' for f in repo_files if re.match(r'checkpoint-\d+/', f)},
    key=lambda folder: int(folder.split('-')[1].strip('/')),
    reverse=True
)

# Exact Phase 1 checkpoint from the repository screenshot.
# We don't silently fall back to an older training step.
if PREFERRED_PHASE1_WEIGHT not in repo_files:
    raise FileNotFoundError(
        f'Expected Phase 1 weights {PREFERRED_PHASE1_WEIGHT!r} in {HF_REPO}. '
        'Check the repository or LOCAL_PHASE1_DIR before training.')

model_file = None
model_folder = None
for folder in list(final_folders) + checkpoint_folders:
    for name in weight_names:
        if folder + name in repo_files:
            model_file = folder + name
            model_folder = folder
            break
    if model_file is not None:
        break
if model_file is None:
    raise FileNotFoundError(
        'No model weights found. The repo must contain extracted model_state.pt '
        'or model.safetensors, not only a backup ZIP. Files: ' + str(repo_files[:30]))
if model_file != PREFERRED_PHASE1_WEIGHT:
    # This also protects against accidentally choosing root model_state.pt
    # when a specific final/checkpoint path was selected.
    raise ValueError(f'Wrong Phase 1 checkpoint selected: {model_file}. '
                     f'Expected: {PREFERRED_PHASE1_WEIGHT}')
print('Using weights:', model_file)

P1_DIR = OUTPUT / 'phase1_files'
# Clear files from an earlier run so we never mix two tokenizers or checkpoints.
if P1_DIR.exists():
    shutil.rmtree(P1_DIR)
P1_DIR.mkdir(parents=True)

def get_file(repo_id, repo_path):
    if source_dir is not None and repo_id == HF_REPO:
        cached = source_dir / repo_path
    else:
        cached = hf_hub_download(repo_id=repo_id, filename=repo_path,
                                 repo_type='model', token=HF_TOKEN)
    destination = P1_DIR / Path(repo_path).name
    shutil.copyfile(cached, destination)
    return destination

weights_path = get_file(HF_REPO, model_file)
weight_file = Path(model_file).name

# Architecture 3 has its config at the repo root, outside checkpoint-6104.
for config_file in (model_folder + 'architecture_config.json',
                    'architecture_config.json'):
    if config_file in repo_files:
        get_file(HF_REPO, config_file)
        break

# Use the same tokenizer that was used to pretrain this model.
tokenizer_folders = [model_folder, '', 'final_model/',
                     'architecture-3-final/', 'architecture-4-final/']
tokenizer_folders.extend(checkpoint_folders)
tok_folder = next((folder for folder in tokenizer_folders
                   if folder + 'tokenizer.json' in repo_files), None)
tokenizer_files = ('tokenizer.json', 'tokenizer_config.json',
                   'special_tokens_map.json', 'added_tokens.json')

if tok_folder is not None:
    for name in tokenizer_files:
        if tok_folder + name in repo_files:
            get_file(HF_REPO, tok_folder + name)
elif LOCAL_TOKENIZER_DIR:
    for name in tokenizer_files:
        source = Path(LOCAL_TOKENIZER_DIR) / name
        if source.exists():
            shutil.copyfile(source, P1_DIR / name)
elif TOKENIZER_REPO_NAME:
    other_repo = HF_USER + '/' + TOKENIZER_REPO_NAME
    other_files = api.list_repo_files(other_repo, repo_type='model')
    other_folder = next((x for x in ('', 'final_model/')
                         if x + 'tokenizer.json' in other_files), None)
    if other_folder is None:
        raise FileNotFoundError('Tokenizer missing from ' + other_repo)
    for name in tokenizer_files:
        if other_folder + name in other_files:
            get_file(other_repo, other_folder + name)

if not (P1_DIR / 'tokenizer.json').is_file():
    raise FileNotFoundError('Tokenizer missing. Upload tokenizer.json or set LOCAL_TOKENIZER_DIR.')

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    str(P1_DIR), pad_token='<pad>', unk_token='<unk>',
    bos_token='<s>', eos_token='</s>'
)
if not tokenizer.is_fast:
    raise RuntimeError('A fast tokenizer is required for character offsets.')

config_path = P1_DIR / 'architecture_config.json'
saved_cfg = json.loads(config_path.read_text()) if config_path.is_file() else {}
if 'vocab_size' in saved_cfg and len(tokenizer) != saved_cfg['vocab_size']:
    raise ValueError('Phase 1 vocabulary and tokenizer sizes differ.')

# Phase 1 model ka checkpoint hi architecture sizes ka source hai.
# Source notebook ke default sizes checkpoint se alag ho sakte hain.
# Safely read the pretrained checkpoint before creating the model.
if weight_file.endswith('.safetensors'):
    from safetensors.torch import load_file
    p1_state = load_file(str(weights_path))
else:
    p1_state = torch.load(weights_path, map_location='cpu', weights_only=True)

if isinstance(p1_state, dict) and 'model_state' in p1_state:
    p1_state = p1_state['model_state']
if not isinstance(p1_state, dict) or 'token_embedding.weight' not in p1_state:
    raise ValueError('Phase 1 checkpoint state_dict missing token_embedding.weight')

# DataParallel ya torch.compile se prefix aaye toh remove karo.
for prefix in ('module.', '_orig_mod.'):
    keys = list(p1_state)
    if keys and all(k.startswith(prefix) for k in keys):
        p1_state = {k[len(prefix):]:v for k,v in p1_state.items()}

checkpoint_vocab, checkpoint_dim = p1_state['token_embedding.weight'].shape
if checkpoint_vocab != len(tokenizer):
    raise ValueError(f'Wrong tokenizer: checkpoint vocab={checkpoint_vocab}, tokenizer={len(tokenizer)}')

# State dict se layer count aur feed-forward size accurately milte hain.
def layer_count(prefix):
    nums = {int(m.group(1)) for k in p1_state
            if (m := re.match(re.escape(prefix) + r'\.(\d+)\.', k))}
    if not nums or nums != set(range(max(nums) + 1)):
        raise ValueError(f'Non-contiguous/missing Phase 1 {prefix}: {sorted(nums)}')
    return len(nums)

def config_heads(arch, dim, layers):
    # First use explicit setting, if one is provided below.
    explicit = {'a1': A1_ATTENTION_HEADS, 'a2': A2_ATTENTION_HEADS}.get(arch)
    if explicit is not None:
        return int(explicit)

    # The checkpoint does not store the number of attention heads.
    # Trust saved architecture_config.json ONLY if its dimensions match.
    saved_dim = saved_cfg.get('d_model', saved_cfg.get('dim'))
    saved_layers = saved_cfg.get('n_encoder_layers', saved_cfg.get('num_layers', saved_cfg.get('n_layers')))
    if (saved_cfg.get('n_heads') is not None or saved_cfg.get('num_heads') is not None):
        if (saved_dim is None or int(saved_dim) == dim) and (saved_layers is None or int(saved_layers) == layers):
            return int(saved_cfg.get('n_heads', saved_cfg.get('num_heads')))

    # A2 repo also has the original notebook. Use its N_HEAD only when
    # N_EMBD/N_LAYER match the actual checkpoint, otherwise it may be stale.
    if arch == 'a2' and 'architecture2.ipynb' in repo_files:
        nb_path = get_file(HF_REPO, 'architecture2.ipynb')
        try:
            nb = json.loads(nb_path.read_text(encoding='utf-8'))
            for cell in nb.get('cells', []):
                src = ''.join(cell.get('source', []))
                found = {key:int(m.group(1)) for key in ('N_EMBD','N_LAYER','N_HEAD')
                         if (m := re.search(r'^\s*' + key + r'\s*=\s*(\d+)', src, flags=re.M))}
                if (found.get('N_EMBD') == dim and found.get('N_LAYER') == layers
                        and found.get('N_HEAD', 0) > 0):
                    return found['N_HEAD']
        except (ValueError, UnicodeError, KeyError) as exc:
            print('Could not read the saved A2 notebook settings:', exc)

    # Default values only valid when checkpoint exactly matches the original source.
    originals = {'a1': (640, 8, 10), 'a2': (600, 8, 15)}
    if arch in originals and (dim, layers) == originals[arch][:2]:
        return originals[arch][2]
    raise ValueError(
        f'{arch.upper()} checkpoint has dim={dim}, layers={layers}. Attention-head count '
        f'cannot be determined from model weights. Please set '
        f'{arch.upper()}_ATTENTION_HEADS in Cell 2 to the Phase 1 value. '
        'Do not guess: wrong head counts may load but change model behavior.')

if ARCH == 'a1':
    n_layers = layer_count('blocks')
    p1_params = dict(vocab_size=checkpoint_vocab,
        context_length=int(p1_state['position_embedding.weight'].shape[0]),
        d_model=checkpoint_dim, n_layers=n_layers,
        n_heads=config_heads('a1',checkpoint_dim,n_layers),
        d_ff=int(p1_state['blocks.0.feed_forward.0.weight'].shape[0]),dropout=0.1)
    if checkpoint_dim % p1_params['n_heads']:
        raise ValueError('A1 embedding size not divisible by attention heads')
elif ARCH == 'a2':
    n_enc, n_dec = layer_count('encoder_blocks'),layer_count('decoder_blocks')
    p1_params = dict(vocab_size=checkpoint_vocab,
        source_length=int(p1_state['source_position.weight'].shape[0]),
        target_length=int(p1_state['target_position.weight'].shape[0]),
        d_model=checkpoint_dim,n_heads=config_heads('a2',checkpoint_dim,n_enc),
        n_encoder_layers=n_enc,n_decoder_layers=n_dec,
        d_ff=int(p1_state['encoder_blocks.0.feed_forward.0.weight'].shape[0]),dropout=0.1)
    if checkpoint_dim % p1_params['n_heads']:
        raise ValueError('A2 embedding size not divisible by attention heads')
elif ARCH == 'a3':
    # RoPE/GQA head counts and local-window sizes cannot be inferred from state_dict.
    need = ('num_heads','num_kv_heads','window_size','chunk_size')
    if any(k not in saved_cfg for k in need):
        raise ValueError('A3 architecture_config.json is missing required attention settings: '
                         + str([k for k in need if k not in saved_cfg]))
    n_layers=layer_count('blocks')
    p1_params=dict(vocab_size=checkpoint_vocab,dim=checkpoint_dim,num_layers=n_layers,
        num_heads=int(saved_cfg['num_heads']),num_kv_heads=int(saved_cfg['num_kv_heads']),
        ffn_dim=int(p1_state['blocks.0.ffn.gate_proj.weight'].shape[0]),
        window_size=int(saved_cfg['window_size']),chunk_size=int(saved_cfg['chunk_size']),
        dropout=float(saved_cfg.get('dropout',0.1)))
    if saved_cfg.get('dim',checkpoint_dim)!=checkpoint_dim or saved_cfg.get('num_layers',n_layers)!=n_layers:
        raise ValueError('A3 architecture_config.json disagrees with checkpoint size/depth')
    if checkpoint_dim % p1_params['num_heads'] or p1_params['num_heads'] % p1_params['num_kv_heads']:
        raise ValueError('A3 invalid GQA query/KV head configuration')
    if (checkpoint_dim // p1_params['num_heads']) % 2:
        raise ValueError('A3 RoPE requires even head dimension')
elif ARCH == 'a4':
    # Dilated masks/patterns are not stored in weights; load saved patterns.
    if 'attention_patterns' not in saved_cfg or 'num_heads' not in saved_cfg:
        raise ValueError('A4 architecture_config.json missing attention_patterns/num_heads')
    n_layers=layer_count('blocks')
    patterns=saved_cfg['attention_patterns']
    if len(patterns)!=n_layers:
        raise ValueError('A4 pattern count does not match checkpoint layer count')
    p1_params=dict(vocab_size=checkpoint_vocab,
        context_length=int(p1_state['position_embedding.weight'].shape[0]),
        dim=checkpoint_dim,num_heads=int(saved_cfg['num_heads']),
        ffn_dim=int(p1_state['blocks.0.ffn.0.weight'].shape[0]),
        attention_patterns=patterns,dropout=float(saved_cfg.get('dropout',0.1)))
    if saved_cfg.get('dim',checkpoint_dim)!=checkpoint_dim:
        raise ValueError('A4 architecture_config.json disagrees with checkpoint width')
    if checkpoint_dim % p1_params['num_heads']:
        raise ValueError('A4 embedding size not divisible by attention heads')
else:
    raise ValueError('Unknown architecture: ' + ARCH)

# Instantiate the ACTUAL pretrained architecture, not the notebook defaults.
def new_backbone(num_layers=None):
    args = dict(p1_params)
    if num_layers is not None:
        if ARCH == 'a3':
            args['num_layers'] = int(num_layers)
        elif ARCH == 'a1':
            args['n_layers'] = int(num_layers)
        elif ARCH == 'a2':
            args['n_encoder_layers'] = int(num_layers)
        else:
            args['attention_patterns'] = args['attention_patterns'][:int(num_layers)]
    if ARCH == 'a1': return DecoderTransformerLM(**args)
    if ARCH == 'a2': return EncoderDecoderTransformer(**args)
    if ARCH == 'a3': return ModernLocalTransformerLM(**args)
    if ARCH == 'a4': return MultiScaleDilatedLM(**args)
    raise ValueError('Unknown architecture')

base = new_backbone()
expected = base.state_dict()

# Phase 1 ke output-head names architecture ke hisaab se different hain.
# A1/A2 mein output_layer hai, A3/A4 mein tied lm_head use hota hai.
# Sirf known head aliases handle karo. Baaki weights strictly check honge.
if ARCH in ('a1', 'a2'):
    # Some saved checkpoints contain this extra name from an older save step.
    if 'lm_head.weight' in p1_state and 'lm_head.weight' not in expected:
        if 'output_layer.weight' not in p1_state:
            # Agar actual output layer missing hai, same-shaped LM head use kar sakte hain.
            if tuple(p1_state['lm_head.weight'].shape) != tuple(expected['output_layer.weight'].shape):
                raise ValueError('Extra lm_head.weight does not match the Phase 1 output layer shape')
            p1_state['output_layer.weight'] = p1_state['lm_head.weight']
            print('Using lm_head.weight as the missing output_layer.weight')
        else:
            print('Ignoring extra lm_head.weight; output_layer.weight is already present')
        del p1_state['lm_head.weight']
else:
    # A3/A4 tie the LM output head to token embeddings.
    # Safetensors can sometimes save just one of these two identical tensors.
    for key, other in (('lm_head.weight', 'token_embedding.weight'),
                       ('token_embedding.weight', 'lm_head.weight')):
        if key in expected and key not in p1_state and other in p1_state:
            if tuple(expected[key].shape) == tuple(p1_state[other].shape):
                p1_state[key] = p1_state[other]
                print('Restored tied weight name:', key)

# All non-alias checkpoint keys still have to match the Phase 1 architecture.

missing = sorted(set(expected)-set(p1_state))
unexpected = sorted(set(p1_state)-set(expected))
wrong_shape = [(k,tuple(p1_state[k].shape),tuple(expected[k].shape))
               for k in expected.keys() & p1_state.keys()
               if tuple(p1_state[k].shape)!=tuple(expected[k].shape)]
if missing or unexpected or wrong_shape:
    raise ValueError(f'Phase 1 model mismatch! Missing: {missing[:6]} | '
                     f'Extra: {unexpected[:6]} | Shape errors: {wrong_shape[:6]}')
base.load_state_dict(p1_state,strict=True)
if ARCH=='a1':
    base.output_layer=nn.Identity()  # A1 needs hidden vectors for token NER.

MODEL_DIM = checkpoint_dim
print('Phase 1 output-head check: passed (v4)')
print('Loaded pretrained',ARCH,'from:',model_file)
print('Actual checkpoint dimensions:',p1_params)
print('NER hidden size:',MODEL_DIM)
# Save the exact Phase 1 provenance with this experiment.
# Useful later for the final report and for checking the same tokenizer/weights.
with open(OUTPUT / 'phase1_used.json', 'w') as f:
    json.dump({'architecture': ARCH, 'repo': HF_REPO,
               'weight_file': model_file, 'model_parameters': p1_params,
               'tokenizer_size': len(tokenizer)}, f, indent=2)



Phase 1 source: P-261110044/cs781-a1-checkpoints
Available files: ['.gitattributes', 'CS781_A1_COMPLETE_BACKUP.zip', 'checkpoint-1000/model.safetensors', 'checkpoint-1000/optimizer.pt', 'checkpoint-1000/rng_state.pth', 'checkpoint-1000/scaler.pt', 'checkpoint-1000/scheduler.pt', 'checkpoint-1000/tokenizer.json', 'checkpoint-1000/tokenizer_config.json', 'checkpoint-1000/trainer_state.json', 'checkpoint-1000/training_args.bin', 'checkpoint-1250/model.safetensors']
Using weights: checkpoint-6104/model.safetensors


checkpoint-6104/model.safetensors:   0%|          | 0.00/419M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'TokenizersBackend'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


Phase 1 output-head check: passed (v4)
Loaded pretrained a1 from: checkpoint-6104/model.safetensors
Actual checkpoint dimensions: {'vocab_size': 50000, 'context_length': 2048, 'd_model': 640, 'n_layers': 8, 'n_heads': 10, 'd_ff': 2560, 'dropout': 0.1}
NER hidden size: 640


In [4]:
# BIO labels: O = outside, B = start, I = continuation.
LABELS = ['APP','RESP','A.COUNSEL','R.COUNSEL','JUDGE','WIT',
          'AUTH','COURT','STAT','PREC','DATE','CASENO']
TAGS = ['O'] + [p + '-' + lab for lab in LABELS for p in ('B','I')]
TAG_ID = {tag:i for i,tag in enumerate(TAGS)}
DIM = MODEL_DIM  # Checkpoint ke actual hidden size ke according

# LoRMA: frozen W ka output ek low-rank transformation se adapt hota hai.
# This is multiplicative on W's output: (I + BA)Wx, unlike ordinary LoRA.
class LoRMALinear(nn.Module):
    def __init__(self, original):
        super().__init__()
        self.original = original
        self.original.requires_grad_(False)
        self.A = nn.Linear(original.out_features, RANK, bias=False)
        self.B = nn.Linear(RANK, original.out_features, bias=False)
        nn.init.zeros_(self.B.weight)
    def forward(self, x):
        y = self.original(x)
        return y + self.B(self.A(y)) * (ALPHA / RANK)

class Adapter(nn.Module):
    # Pehle dimension chhota, phir original dimension mein wapas.
    def __init__(self, dim):
        super().__init__()
        self.down = nn.Linear(dim, 64)
        self.up = nn.Linear(64, dim)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return self.up(F.gelu(self.down(x)))

class NERModel(nn.Module):
    def __init__(self, base, method):
        super().__init__()
        self.base = base
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(DIM, len(TAGS))

        if method not in ('full', 'kd'):
            self.base.requires_grad_(False)
        if method == 'partial':
            # A2 ka sirf last 2 encoder blocks train honge.
            assert ARCH == 'a2'
            for block in self.base.encoder_blocks[-2:]:
                block.requires_grad_(True)
        elif method == 'adapters':
            # A1 ke har block ke baad chhota residual adapter lagaya.
            assert ARCH == 'a1'
            self.adapters = nn.ModuleList([Adapter(DIM) for _ in self.base.blocks])
            for block, adapter in zip(self.base.blocks, self.adapters):
                block.register_forward_hook(lambda m, inp, out, ad=adapter: out + ad(out))
        elif method == 'lorma':
            # A4 attention ke q/v projection mein LoRMA lagaya.
            assert ARCH == 'a4'
            for block in self.base.blocks:
                block.attn.q_proj = LoRMALinear(block.attn.q_proj)
                block.attn.v_proj = LoRMALinear(block.attn.v_proj)

    def forward(self, input_ids):
        if ARCH == 'a1':
            hidden = self.base(input_ids=input_ids).logits  # LM output_layer = Identity
        elif ARCH == 'a2':
            hidden = self.base.encode(input_ids)  # Encoder output; decoder NER ke liye nahi
        else:
            hidden, _ = self.base._run_hidden(input_ids)
        return self.classifier(self.dropout(hidden))

if TECHNIQUE == 'kd':
    # Phase 1 ke first layers se student banta hai, random weights nahi.
    assert ARCH == 'a3'
    student_base = new_backbone(num_layers=max(1, min(4, p1_params['num_layers'] - 1)))
    student_state = student_base.state_dict()
    for key in student_state:
        if key not in p1_state or student_state[key].shape != p1_state[key].shape:
            raise ValueError('Missing/incompatible student weight: ' + key)
        student_state[key] = p1_state[key]
    student_base.load_state_dict(student_state, strict=True)
    base = student_base
    del student_state, student_base

del p1_state
model = NERModel(base, TECHNIQUE)

# No LoRA wrapper for other methods. Their trainable parameters differ genuinely.
if TECHNIQUE == 'lora':
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=RANK, lora_alpha=ALPHA, lora_dropout=0.05,
        target_modules=['q_proj','v_proj'], modules_to_save=['classifier'], bias='none'))
model = model.to(DEVICE)
num_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
num_total = sum(p.numel() for p in model.parameters())
print('Architecture:', ARCH, '| method:', TECHNIQUE)
print('Trainable parameters:', num_train, '| total:', num_total)
assert num_train > 0
if TECHNIQUE in ('adapters','lorma','partial'):
    assert num_train < num_total, 'Expected frozen backbone parameters'
assert model.classifier.in_features == MODEL_DIM


Architecture: a1 | method: adapters
Trainable parameters: 677017 | total: 73377177


In [5]:

# Official Phase-2 data only. Use fold1+fold2, then split by document ID.
# No test annotations are downloaded or used for validation.
def official_file(filename):
    return hf_hub_download(repo_id=DATA_REPO,
        filename='Phase_2_LNER/'+filename,repo_type='dataset',token=HF_TOKEN)
def read_official_json(path):
    with open(official_file(path),encoding='utf-8') as f: return json.load(f)

def extract_documents(obj):
    if isinstance(obj,dict):
        result={}
        for docid,v in obj.items():
            if isinstance(v,str): result[str(docid)]=v
            elif isinstance(v,dict):
                val=v.get('text',v.get('document',v.get('content')))
                if isinstance(val,list): val='\n'.join(val)
                if not isinstance(val,str): raise ValueError('Unexpected document text shape for '+str(docid))
                result[str(docid)]=val
            else: raise ValueError('Unexpected document value type for '+str(docid))
        return result
    if isinstance(obj,list):
        out={}
        for row in obj:
            docid=str(row.get('id',row.get('doc_id',row.get('document_id'))))
            val=row.get('text',row.get('document',row.get('content')))
            if isinstance(val,list): val='\n'.join(val)
            if not isinstance(val,str) or docid=='None':raise ValueError('Unexpected document row')
            if docid in out:raise ValueError('Duplicate document ID: '+docid)
            out[docid]=val
        return out
    raise ValueError('Unrecognized official text file format')

def one_entity(item):
    # Accept the official [start,end,label,text] form plus dictionary equivalents.
    if isinstance(item,(list,tuple)) and len(item)>=3:
        return (int(item[0]),int(item[1]),str(item[2]),str(item[3]) if len(item)>3 else None)
    if isinstance(item,dict):
        if 'span' in item and isinstance(item['span'],(list,tuple)):
            a,b=item['span'][:2]
        else:
            a=item.get('start',item.get('start_offset',item.get('start_char')))
            b=item.get('end',item.get('end_offset',item.get('end_char')))
        label=item.get('label',item.get('type',item.get('entity_type')))
        ent_text=item.get('text',item.get('entity_text'))
        if a is None or b is None or label is None:
            raise ValueError('Unrecognized annotation fields: '+str(item)[:250])
        return int(a),int(b),str(label),ent_text
    raise ValueError('Unrecognized entity format '+str(item)[:200])

def extract_annotations(gold):
    if isinstance(gold,list):
        gold={str(r.get('id',r.get('doc_id'))):r for r in gold}
    if not isinstance(gold,dict):raise ValueError('Unexpected gold JSON structure')
    out={}
    for docid, row in gold.items():
        if isinstance(row,dict):
            row=row.get('entities',row.get('annotations',row.get('spans',row.get('labels'))))
        if row is None: row=[]
        if not isinstance(row,list):raise ValueError('Gold annotations not a list for '+str(docid))
        out[str(docid)]=[one_entity(e) for e in row]
    return out

texts={}
golds={}
for fold in ('fold1','fold2'):
    tt=extract_documents(read_official_json('Train/'+fold+'-text.json'))
    gg=extract_annotations(read_official_json('Train/'+fold+'-gold.json'))
    assert tt.keys()==gg.keys(),f'Missing/extra IDs in {fold}: {len(tt)} texts, {len(gg)} golds'
    assert texts.keys().isdisjoint(tt), 'Duplicate IDs across training folds'
    texts.update(tt); golds.update(gg)

# Confirm label list agrees with the competition document (12 categories).
with open(official_file('Train/labels.txt'),encoding='utf-8') as f:
    official_labels=[line.strip() for line in f if line.strip()]
assert set(official_labels)==set(LABELS), ('Unexpected label names: ',official_labels)

warn_count=0
for docid, spans in golds.items():
    t=texts[docid]
    for a,b,lab,entity_text in spans:
        assert 0<=a<b<=len(t) and lab in LABELS,(docid,a,b,lab)
        if entity_text is not None and t[a:b]!=entity_text:
            # Some annotation sets may have mismatched auxiliary text; offsets/labels are authoritative.
            if warn_count<5: print('WARNING annotation text differs from text at offsets:',docid,repr(t[a:b]),repr(entity_text))
            warn_count+=1

ids=sorted(texts)
random.Random(SEED).shuffle(ids)
val_n=max(1,round(len(ids)*VAL_PART))
dev_ids=ids[:val_n]; train_ids=ids[val_n:]
if MAX_TRAIN_DOCS is not None:train_ids=train_ids[:MAX_TRAIN_DOCS]
if MAX_VAL_DOCS is not None:dev_ids=dev_ids[:MAX_VAL_DOCS]
assert set(train_ids).isdisjoint(dev_ids)
print('Train documents:',len(train_ids),'held-out documents:',len(dev_ids))
print('Training labels:',Counter(s[2] for did in train_ids for s in golds[did]))


fold1-text.json: 0.00B [00:00, ?B/s]

fold1-gold.json: 0.00B [00:00, ?B/s]

fold2-text.json: 0.00B [00:00, ?B/s]

fold2-gold.json: 0.00B [00:00, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Train documents: 63 held-out documents: 7
Training labels: Counter({'DATE': 1420, 'STAT': 1208, 'PREC': 1102, 'AUTH': 976, 'COURT': 832, 'CASENO': 697, 'APP': 390, 'RESP': 317, 'JUDGE': 203, 'WIT': 187, 'A.COUNSEL': 179, 'R.COUNSEL': 161})


In [6]:

# Byte/character offset safety: tokenize the ORIGINAL full document (no normalization).
# Every offset returned by the fast tokenizer points into the original source string.
# Replace the existing encode_document() function with this one.
# It uses the SAME tokenizer trained in Phase 1.
# The change is only in matching character spans to BPE tokens.

def encode_document(text, spans=None):
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
        return_attention_mask=False
    )
    input_ids = encoded['input_ids']
    offsets = [tuple(pair) for pair in encoded['offset_mapping']]
    tags = [TAG_ID['O']] * len(input_ids)

    if spans is not None:
        used_tokens = set()

        # Sort by start character. Longer overlapping entities go first.
        for a, b, label, _ in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
            # Include BPE tokens that overlap the gold entity.
            # E.g. token ' S' has offsets (11,13), but gold starts at 12.
            token_idxs = [
                i for i, (start, end) in enumerate(offsets)
                if end > start and start < b and end > a
            ]
            if not token_idxs:
                continue

            # One token cannot represent two labels at the same time.
            if any(i in used_tokens for i in token_idxs):
                continue

            first_start = offsets[token_idxs[0]][0]
            last_end = offsets[token_idxs[-1]][1]

            # Crossing the gold boundary is okay if ONLY whitespace is outside.
            # If there are other letters/punctuation outside, strict offsets
            # cannot be recovered from a plain token BIO label.
            left_extra = text[first_start:a] if first_start < a else ''
            right_extra = text[b:last_end] if last_end > b else ''
            if left_extra.strip() or right_extra.strip():
                continue

            for k, i in enumerate(token_idxs):
                prefix = 'B-' if k == 0 else 'I-'
                tags[i] = TAG_ID[prefix + label]
                used_tokens.add(i)

    return input_ids, offsets, tags
def make_windows(input_ids,offsets,tag_ids=None):
    result=[]
    for begin in range(0,len(input_ids),STRIDE):
        end=min(begin+MAX_LEN,len(input_ids))
        if begin>=end:break
        window={'ids':input_ids[begin:end], 'offsets':offsets[begin:end], 'begin':begin}
        if tag_ids is not None:window['tags']=tag_ids[begin:end]
        result.append(window)
        if end==len(input_ids):break
    return result

class LNERWindows(Dataset):
    def __init__(self,doc_ids):
        self.examples=[]
        for did in doc_ids:
            a,b,c=encode_document(texts[did],golds[did])
            for row in make_windows(a,b,c):
                self.examples.append(row)
    def __len__(self):return len(self.examples)
    def __getitem__(self,i):return self.examples[i]

def collate_one(batch):
    assert len(batch)==1, 'Use train batch size 1; A2 encoder does not mask padding.'
    row=batch[0]
    return (torch.tensor([row['ids']],dtype=torch.long),
            torch.tensor([row['tags']],dtype=torch.long))

train_ds=LNERWindows(train_ids)
loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,
                  collate_fn=collate_one,num_workers=0)
print('Train windows:',len(train_ds))
assert len(train_ds)>0

# Training se pehle ek tiny forward pass check kar lo.
# Dimensional errors ko pehle hi catch karna better hai.
with torch.no_grad():
    tiny = torch.tensor([train_ds[0]['ids'][:min(8,len(train_ds[0]['ids']))]],
                        dtype=torch.long,device=DEVICE)
    model.eval()
    tiny_logits = model(tiny)
    assert tiny_logits.shape == (1,tiny.size(1),len(TAGS)), tiny_logits.shape
    assert torch.isfinite(tiny_logits).all(), 'Non-finite logits on small test'
    model.train()
print('Pre-training forward check passed:',tuple(tiny_logits.shape))


Train windows: 2182
Pre-training forward check passed: (1, 8, 25)


In [7]:
# Validation and prediction: pehle windows jodo, phir full BIO sequence decode karo.
from tqdm.auto import tqdm
import numpy as np

# Same offset conversion as before. For greedy tags, an invalid I tag starts a new entity.
def spans_from_tags(text, offsets, tag_ids, confidences=None):
    if confidences is None:
        confidences = [1.0] * len(tag_ids)
    spans = []
    current = None

    def close():
        nonlocal current
        if current is not None:
            a, b, label, conf = current
            # Original text ke start/end mein sirf extra spaces trim karo.
            piece = text[a:b]
            a += len(piece) - len(piece.lstrip())
            b -= len(piece) - len(piece.rstrip())
            if a < b:
                spans.append((a, b, label, sum(conf) / len(conf)))
            current = None

    for (s, e), tag_id, prob in zip(offsets, tag_ids, confidences):
        if not (0 <= s < e <= len(text)):
            close()
            continue
        tag = TAGS[int(tag_id)]
        if tag == 'O':
            close()
            continue
        prefix, label = tag.split('-', 1)
        if (current and prefix == 'I' and label == current[2]
                and not text[current[1]:s].strip()):
            current = (current[0], e, label, current[3] + [float(prob)])
        else:
            close()
            current = (s, e, label, [float(prob)])
    close()
    return spans


def prefer_longer_context(merged, best_left, start, window_logits):
    # A1/A3/A4 causal hain: zyada left context wala window use karo.
    # A2 bidirectional encoder hai: left aur right dono context important hain.
    n = len(window_logits)
    pos = torch.arange(n, dtype=torch.long)
    context = torch.minimum(pos, n - 1 - pos) if ARCH == 'a2' else pos
    old = best_left[start:start+n]
    replace = context > old
    merged[start:start+n][replace] = window_logits[replace]
    old[replace] = context[replace]


def valid_bio_viterbi(logits):
    # Highest scoring BIO sequence dhundho, but invalid I transitions allow mat karo.
    # I-X ke just pehle B-X ya I-X hona chahiye.
    scores = logits.detach().float().cpu().numpy()
    length, n_tags = scores.shape
    if length == 0:
        return []

    b_ids = np.array([TAG_ID['B-' + lab] for lab in LABELS])
    i_ids = np.array([TAG_ID['I-' + lab] for lab in LABELS])
    parents = np.zeros((length, n_tags), dtype=np.int16)

    dp = scores[0].copy()
    dp[i_ids] = -1e9  # Sentence/document I-X se start nahi ho sakta

    for t in range(1, length):
        best_prev = int(dp.argmax())
        next_dp = scores[t] + dp[best_prev]
        parents[t, :] = best_prev

        # I-X can only continue B-X or I-X. B and O can follow any tag.
        take_b = dp[b_ids] >= dp[i_ids]
        valid_prev = np.where(take_b, b_ids, i_ids)
        next_dp[i_ids] = scores[t, i_ids] + dp[valid_prev]
        parents[t, i_ids] = valid_prev
        dp = next_dp

    # Backtrack from the last token to get final tag IDs.
    tags = [0] * length
    tags[-1] = int(dp.argmax())
    for t in range(length - 1, 0, -1):
        tags[t - 1] = int(parents[t, tags[t]])
    return tags


@torch.no_grad()
def predict_one_doc_legacy(document):
    # Purana method: each window ka span alag banao, confidence se merge karo.
    # Sirf comparison ke liye rakha hai. Final prediction neeche wala method hai.
    model.eval()
    ids, offsets, _ = encode_document(document, None)
    candidate = {}
    for window in make_windows(ids, offsets):
        inp = torch.tensor([window['ids']], dtype=torch.long, device=DEVICE)
        probabilities = model(inp)[0].softmax(dim=-1)
        confidence, tags = probabilities.max(dim=-1)
        spans = spans_from_tags(document, window['offsets'],
                                tags.cpu().tolist(), confidence.cpu().tolist())
        for a, b, label, conf in spans:
            key = (a, b, label)
            candidate[key] = max(conf, candidate.get(key, 0.0))
    ranked = sorted(candidate.items(), key=lambda row: (-row[1], -(row[0][1] - row[0][0])))
    chosen = []
    for (a, b, label), _ in ranked:
        if not any(a < end and start < b for start, end, _ in chosen):
            chosen.append((a, b, label))
    return sorted(chosen, key=lambda x: (x[0], x[1], x[2]))


@torch.no_grad()
def predict_one_doc(document, method=None):
    if method is None:
        method = DECODE_METHOD
    if method == 'legacy':
        return predict_one_doc_legacy(document)
    if method not in ('greedy', 'viterbi'):
        raise ValueError('Use legacy, greedy or viterbi')

    model.eval()
    ids, offsets, _ = encode_document(document, None)
    if not ids:
        return []

    # Full document ki logits yahan store kar rahe hain (CPU memory).
    merged = torch.empty((len(ids), len(TAGS)), dtype=torch.float32)
    best_left = torch.full((len(ids),), -1, dtype=torch.long)

    for window in make_windows(ids, offsets):
        inp = torch.tensor([window['ids']], dtype=torch.long, device=DEVICE)
        window_logits = model(inp)[0].detach().float().cpu()
        if not torch.isfinite(window_logits).all():
            raise FloatingPointError('Non-finite NER logits: check training/checkpoint')
        prefer_longer_context(merged, best_left, window['begin'], window_logits)

    assert (best_left >= 0).all(), 'Some document tokens were not covered by windows'

    if method == 'viterbi':
        tag_ids = valid_bio_viterbi(merged)
    else:
        tag_ids = merged.argmax(dim=-1).tolist()

    # Now decode exactly once using original full-document offsets.
    spans = spans_from_tags(document, offsets, tag_ids)
    return [(a, b, label) for a, b, label, _ in spans]


def strict_scores(document_ids, method=None):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in tqdm(document_ids, desc='Validation', leave=False):
        predicted = set(predict_one_doc(texts[did], method=method))
        actual = {(a, b, label) for a, b, label, _ in golds[did]}
        for _, _, label in predicted & actual: tp[label] += 1
        for _, _, label in predicted - actual: fp[label] += 1
        for _, _, label in actual - predicted: fn[label] += 1

    scores, precisions, recalls = [], [], []
    for label in LABELS:
        precision = tp[label] / max(1, tp[label] + fp[label])
        recall = tp[label] / max(1, tp[label] + fn[label])
        f1 = 2 * precision * recall / max(1e-12, precision + recall)
        scores.append(f1)
        precisions.append(precision)
        recalls.append(recall)
    return {
        'strict_macro_f1': float(np.mean(scores)),
        'strict_macro_precision': float(np.mean(precisions)),
        'strict_macro_recall': float(np.mean(recalls)),
        'per_label_f1': dict(zip(LABELS, map(float, scores)))
    }

# Simple tests: no model or GPU needed.
assert spans_from_tags('The Delhi High Court sat.', [(4,9), (10,14), (15,20)],
       [TAG_ID['B-COURT'], TAG_ID['I-COURT'], TAG_ID['I-COURT']]) == [(4,20,'COURT',1.0)]
assert spans_from_tags('ABC', [(0,1)], [TAG_ID['O']]) == []
assert max(len(w['ids']) for w in make_windows(list(range(400)), [(0,1)]*400)) <= MAX_LEN

# First token ko I-COURT ka high score do. Viterbi should still return valid BIO.
toy = torch.full((2, len(TAGS)), -5.0)
toy[0, TAG_ID['I-COURT']] = 8.0
toy[0, TAG_ID['B-COURT']] = 7.0
toy[1, TAG_ID['I-COURT']] = 8.0
assert [TAGS[i] for i in valid_bio_viterbi(toy)] == ['B-COURT', 'I-COURT']

# Overlap mein later window ko prefer karna chahiye, because more left context.
toy_merge = torch.empty((4, len(TAGS)))
toy_left = torch.full((4,), -1, dtype=torch.long)
prefer_longer_context(toy_merge, toy_left, 0, torch.ones((3, len(TAGS))))
prefer_longer_context(toy_merge, toy_left, 1, torch.full((3, len(TAGS)), 2.0))
assert toy_merge[:, 0].tolist() == ([1.0, 1.0, 2.0, 2.0] if ARCH == 'a2' else [1.0, 1.0, 1.0, 2.0])
print('Stitching and valid-BIO unit tests passed')


Stitching and valid-BIO unit tests passed


In [8]:
# Check token/entity alignment BEFORE training.
# This is important: in an earlier A3 run only 4 of 139 gold spans
# could be reconstructed. More epochs cannot fix bad offsets.
# These checks use the TRUE labels; no GPU or model predictions are needed.
from collections import Counter

def check_alignment(doc_ids, show_examples=True):
    total_gold = 0
    total_recovered = 0
    mismatched_text = 0
    tag_counts = Counter()
    missing_examples = []

    for docid in doc_ids:
        doc = texts[docid]
        gold_spans = golds[docid]
        ids, offsets, tags = encode_document(doc, gold_spans)
        tag_counts.update(TAGS[tag] for tag in tags)
        correct = {(a, b, lab) for a, b, lab, _ in gold_spans}
        rebuilt = {(a, b, lab) for a, b, lab, _ in spans_from_tags(doc, offsets, tags)}
        total_gold += len(correct)
        total_recovered += len(correct & rebuilt)

        for a, b, lab, entity_text in gold_spans:
            if entity_text is not None and doc[a:b] != entity_text:
                mismatched_text += 1
        if show_examples:
            for a, b, lab in sorted(correct - rebuilt):
                if len(missing_examples) < 5:
                    missing_examples.append((lab, (a, b), repr(doc[a:b])[:90]))

    coverage = total_recovered / max(1, total_gold)
    print('Documents checked:', len(doc_ids))
    print('Gold spans:', total_gold)
    print('Gold spans reconstructed from TRUE BIO tags:', total_recovered)
    print('Exact gold-span coverage:', f'{coverage:.2%}')
    print('Annotation text mismatches:', mismatched_text)
    print('Most common BIO labels:', tag_counts.most_common(8))
    if show_examples:
        print('Examples missing after tokenization:', missing_examples)
    if coverage < 0.8:
        print('WARNING: Serious token/character alignment problem.')
        print('Do not assume that extra epochs or a longer MAX_LEN will fix this.')
    return coverage

# Find out WHY an annotation is missing from the BIO labels.
# This is a short summary; it cannot by itself correct boundary problems.
def count_skip_reasons(doc_ids):
    reasons = Counter()
    samples = []
    for did in doc_ids:
        doc = texts[did]
        _, offsets, _ = encode_document(doc, golds[did])
        occupied = set()
        for a, b, lab, _ in sorted(golds[did], key=lambda x: (x[0], -(x[1]-x[0]))):
            touching = [i for i, (s, e) in enumerate(offsets)
                        if e > s and s < b and e > a]
            if not touching:
                why = 'no_token_touching_span'
            elif occupied.intersection(touching):
                why = 'overlapping_annotations'
            else:
                first = offsets[touching[0]][0]
                last = offsets[touching[-1]][1]
                left = doc[first:a] if first < a else ''
                right = doc[b:last] if last > b else ''
                if left.strip() or right.strip():
                    why = 'non_whitespace_crosses_boundary'
                else:
                    why = 'accepted_by_encoder'
                    occupied.update(touching)
            reasons[why] += 1
            if why != 'accepted_by_encoder' and len(samples) < 5:
                samples.append((why, lab, repr(doc[a:b])[:65]))
    print('Why annotations are kept/skipped:', dict(reasons))
    print('Skipped entity examples:', samples)

# Check at least a few documents, not only the first validation document.
val_alignment = check_alignment(dev_ids[:min(5, len(dev_ids))])
train_alignment = check_alignment(train_ids[:min(5, len(train_ids))], show_examples=False)
count_skip_reasons(dev_ids[:min(5, len(dev_ids))])

# Set this to True if you want to stop instead of training on labels
# that cannot be recovered exactly. Leave False only for debugging.
STOP_IF_ALIGNMENT_BAD = False       # CHANGE to True before a serious run
if STOP_IF_ALIGNMENT_BAD and min(val_alignment, train_alignment) < 0.8:
    raise ValueError('Bad gold-span alignment. Fix preprocessing before full training.')


# A second pre-training test: entity spans should also survive the sliding windows.
# This is separate from the full-document tokenizer alignment check above.
def check_window_alignment(doc_ids):
    n_gold = 0
    n_found = 0
    missed_examples = []
    for did in doc_ids:
        doc = texts[did]
        ids, offsets, gold_tags = encode_document(doc, golds[did])
        target = {(a,b,lab) for a,b,lab,_ in golds[did]}
        decoded = set()
        for window in make_windows(ids, offsets, gold_tags):
            for a,b,lab,_ in spans_from_tags(doc, window['offsets'], window['tags']):
                decoded.add((a,b,lab))
        n_gold += len(target)
        n_found += len(target & decoded)
        for a,b,lab in sorted(target-decoded):
            if len(missed_examples) < 5:
                missed_examples.append((lab,(a,b),repr(doc[a:b])[:70]))
    print('Gold spans surviving sliding-window BIO decoding:', n_found, '/', n_gold)
    print('Window-level exact coverage:', f'{n_found / max(1,n_gold):.2%}')
    print('Window-level examples missed:', missed_examples)
    if n_gold and n_found/n_gold < 0.9:
        print('WARNING: window splitting is losing many spans. Review STRIDE and MAX_LEN.')

check_window_alignment(dev_ids[:min(5,len(dev_ids))])


Documents checked: 5
Gold spans: 488
Gold spans reconstructed from TRUE BIO tags: 483
Exact gold-span coverage: 98.98%
Annotation text mismatches: 0
Most common BIO labels: [('O', 21702), ('I-DATE', 558), ('I-CASENO', 388), ('I-PREC', 208), ('I-COURT', 204), ('B-DATE', 139), ('B-COURT', 135), ('I-STAT', 85)]
Examples missing after tokenization: [('AUTH', (1963, 1976), "'Railway Board'"), ('AUTH', (11218, 11231), "'Railway Board'"), ('PREC', (12849, 12901), "'Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]'"), ('PREC', (14121, 14174), "'Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]'"), ('COURT', (19254, 19264), "'High Court'")]
Documents checked: 5
Gold spans: 702
Gold spans reconstructed from TRUE BIO tags: 694
Exact gold-span coverage: 98.86%
Annotation text mismatches: 0
Most common BIO labels: [('O', 30652), ('I-PREC', 705), ('I-DATE', 512), ('I-STAT', 185), ('I-CASENO', 176), ('B-DATE', 127), ('I-AUTH', 117), ('I-COURT', 115)]
Why annotations are kept/skipped: {'accept

In [9]:
# Class imbalance hai, but O ko pehle jitna low weight nahi de rahe.
weights = torch.ones(len(TAGS), device=DEVICE)
weights[TAG_ID['O']] = O_WEIGHT

# Head naya hai; backbone/adapter ke parameters ko alag LR de rahe hain.
# Full/partial/KD mein backbone update hota hai; adapters/LoRMA mein chhote weights.
head_params, adapt_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if 'classifier' in name:
        head_params.append(param)
    else:
        adapt_params.append(param)
assert head_params and adapt_params, 'Classifier aur adaptation params dono train hone chahiye'
optimizer = torch.optim.AdamW([
    {'params': adapt_params, 'lr': LR_ADAPT},
    {'params': head_params, 'lr': LR_HEAD},
], weight_decay=0.01)
print('Trainable adaptation/backbone params:', sum(p.numel() for p in adapt_params))
print('Trainable classifier params:', sum(p.numel() for p in head_params))
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

teacher = None  # Is experiment me knowledge distillation nahi hai.

# Displaying time here does not change how the model trains.
def format_time(seconds):
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

best_f1=-1.0
best_epoch=0
no_improve=0  # Last best score ke baad kitne epochs improve nahi hue
history=[]
CKPT=OUTPUT / (EXP_ID+'_best.pt')
all_start=time.perf_counter()

for epoch in range(1,EPOCHS+1):
    epoch_start=time.perf_counter()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total=0.0

    # tqdm shows completed batches, speed and approximate time remaining.
    progress=tqdm(enumerate(loader),total=len(loader),
                  desc=f'Epoch {epoch}/{EPOCHS} training')

    for i,(input_ids,tags) in progress:
        input_ids=input_ids.to(DEVICE)
        tags=tags.to(DEVICE)

        with torch.autocast(device_type=DEVICE.type,enabled=(DEVICE.type=='cuda')):
            logits=model(input_ids)
            loss=F.cross_entropy(logits.reshape(-1,len(TAGS)),
                                 tags.reshape(-1),weight=weights)
            if teacher is not None:
                with torch.no_grad():
                    teach_logits=teacher(input_ids)
                distill=F.kl_div(F.log_softmax(logits/KD_TEMP,dim=-1),
                                 F.softmax(teach_logits/KD_TEMP,dim=-1),
                                 reduction='batchmean')
                distill=distill/input_ids.size(1)
                loss=(1-KD_MIX)*loss+KD_MIX*(KD_TEMP**2)*distill
            # Last accumulation group chhota ho toh uski loss bhi sahi divide karo.
            group_start = (i // ACCUM_STEPS) * ACCUM_STEPS
            group_size = min(ACCUM_STEPS, len(loader) - group_start)
            loss = loss / group_size

        scaler.scale(loss).backward()

        if (i+1)%ACCUM_STEPS==0 or i+1==len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total += float(loss.detach()) * group_size

        # Only refresh the loss display sometimes, to keep it lightweight.
        if (i+1)%10==0 or i+1==len(loader):
            progress.set_postfix(loss=f'{total/(i+1):.4f}')

    train_seconds=time.perf_counter()-epoch_start

    # Validation has a separate progress bar.
    val_start=time.perf_counter()
    metrics=strict_scores(dev_ids)
    val_seconds=time.perf_counter()-val_start
    epoch_seconds=time.perf_counter()-epoch_start

    metrics.update({
        'epoch':epoch,
        'mean_training_loss':total/len(loader),
        'train_seconds':round(train_seconds,2),
        'validation_seconds':round(val_seconds,2),
        'epoch_seconds':round(epoch_seconds,2)
    })
    history.append(metrics)

    print(f"Epoch {epoch}/{EPOCHS} | loss {metrics['mean_training_loss']:.4f} | "
          f"strict macro-F1 {metrics['strict_macro_f1']:.4f}")
    print('Train time:',format_time(train_seconds),
          '| Validation time:',format_time(val_seconds),
          '| Total epoch time:',format_time(epoch_seconds))

    with open(OUTPUT/'history.json','w') as f:
        json.dump(history,f,indent=2)

    if metrics['strict_macro_f1']>best_f1:
        # Better val F1 mila toh best model save karo aur patience reset karo.
        best_f1=metrics['strict_macro_f1']
        best_epoch=epoch
        no_improve=0
        torch.save({'model_state':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                    'epoch':epoch,'labels':TAGS,'technique':TECHNIQUE,'architecture':ARCH,
                    'phase1_repo':HF_REPO,'metrics':metrics,'seed':SEED,
                    'train_doc_ids':train_ids,'dev_doc_ids':dev_ids,
                    'max_len':MAX_LEN,'stride':STRIDE,'o_weight':O_WEIGHT,
                    'lr_adapt':LR_ADAPT,'lr_head':LR_HEAD,'decode_method':DECODE_METHOD},CKPT)
        print('Saved best checkpoint:',CKPT)
    else:
        no_improve+=1
        print(f'No F1 improvement: {no_improve}/{PATIENCE} epochs')
        if no_improve>=PATIENCE:
            print(f'Early stopping at epoch {epoch}. Best epoch was {best_epoch}.')
            break

print('Total training + validation time:',format_time(time.perf_counter()-all_start))
print(f'Best held-out macro-F1: {best_f1:.4f} (epoch {best_epoch})')

# Last epoch zaroori nahi ki best ho, isliye saved best weights load kar rahe hain.
best_state = torch.load(CKPT, map_location='cpu', weights_only=True)
model.load_state_dict(best_state['model_state'], strict=True)
model.eval()
print('Best model loaded from epoch:', best_state['epoch'])
del best_state


Trainable adaptation/backbone params: 660992
Trainable classifier params: 16025


Epoch 1/30 training:   0%|          | 0/2182 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 1/30 | loss 0.3605 | strict macro-F1 0.2240
Train time: 00:00:46 | Validation time: 00:00:02 | Total epoch time: 00:00:48
Saved best checkpoint: /kaggle/working/phase2_a1_adapters_v2_30ep/a1_adapters_v2_30ep_best.pt


Epoch 2/30 training:   0%|          | 0/2182 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

... (33 outputs left out)

Epoch 13/30 | loss 0.0620 | strict macro-F1 0.3404
Train time: 00:00:45 | Validation time: 00:00:02 | Total epoch time: 00:00:47
No F1 improvement: 4/5 epochs


Epoch 14/30 training:   0%|          | 0/2182 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 14/30 | loss 0.0565 | strict macro-F1 0.3324
Train time: 00:00:46 | Validation time: 00:00:02 | Total epoch time: 00:00:48
No F1 improvement: 5/5 epochs
Early stopping at epoch 14. Best epoch was 9.
Total training + validation time: 00:10:54
Best held-out macro-F1: 0.3535 (epoch 9)
Best model loaded from epoch: 9


In [10]:
# Optional extra checks after training. Check predicted vs gold BIO labels
# This only prints numbers; it does not change the trained model.
from collections import Counter

print('Gold BIO tags across training windows:')
actual_counts = Counter()
for row in train_ds.examples:
    actual_counts.update(TAGS[t] for t in row['tags'])
print(actual_counts.most_common(15))
print('Entity-token fraction:',
      round(sum(v for t, v in actual_counts.items() if t != 'O') /
            max(1, sum(actual_counts.values())), 5))

# Check a few windows that actually contain gold entities.
examples_with_entities = [row for row in train_ds.examples
                          if any(t != TAG_ID['O'] for t in row['tags'])][:5]
predicted_counts = Counter()
model.eval()
with torch.no_grad():
    for row in examples_with_entities:
        sample = torch.tensor([row['ids']], dtype=torch.long, device=DEVICE)
        output = model(sample)[0].argmax(-1).tolist()
        predicted_counts.update(TAGS[t] for t in output)
print('Predicted BIO tags on 5 entity-containing windows:')
print(predicted_counts.most_common(15))

# Repeat the gold round-trip test on the validation document.
check_alignment(dev_ids[:1])


Gold BIO tags across training windows:
[('O', 506122), ('I-PREC', 12087), ('I-DATE', 7301), ('I-CASENO', 6484), ('I-STAT', 4122), ('B-DATE', 1864), ('I-AUTH', 1595), ('B-STAT', 1574), ('I-COURT', 1462), ('B-PREC', 1386), ('B-AUTH', 1301), ('I-APP', 1076), ('B-COURT', 1066), ('B-CASENO', 882), ('I-RESP', 847)]
Entity-token fraction: 0.08519
Predicted BIO tags on 5 entity-containing windows:
[('O', 1043), ('I-PREC', 121), ('I-DATE', 24), ('I-JUDGE', 22), ('I-STAT', 19), ('B-AUTH', 7), ('I-APP', 6), ('B-DATE', 6), ('I-COURT', 6), ('B-COURT', 5), ('B-JUDGE', 4), ('I-AUTH', 4), ('B-STAT', 4), ('B-PREC', 4), ('I-RESP', 3)]
Documents checked: 1
Gold spans: 139
Gold spans reconstructed from TRUE BIO tags: 139
Exact gold-span coverage: 100.00%
Annotation text mismatches: 0
Most common BIO labels: [('O', 4306), ('I-DATE', 175), ('I-CASENO', 95), ('B-DATE', 44), ('I-JUDGE', 37), ('B-RESP', 36), ('I-APP', 34), ('I-RESP', 34)]
Examples missing after tokenization: []


1.0

In [11]:
# Training ke baad error analysis: model kya predict kar raha hai?
# Strict F1 mein start, end aur label teeno exact hone chahiye.
from collections import Counter

def span_report(name, doc_ids):
    gold_count, pred_count, right_count = Counter(), Counter(), Counter()
    missed, extra = [], []
    for did in doc_ids:
        gold = {(a, b, lab) for a, b, lab, _ in golds[did]}
        pred = set(predict_one_doc(texts[did]))
        right = gold & pred
        gold_count.update(x[2] for x in gold)
        pred_count.update(x[2] for x in pred)
        right_count.update(x[2] for x in right)
        missed.extend((did, x) for x in gold - pred)
        extra.extend((did, x) for x in pred - gold)

    per_class = {}
    for lab in LABELS:
        g, p, tp = gold_count[lab], pred_count[lab], right_count[lab]
        precision = tp / p if p else 0.0
        recall = tp / g if g else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[lab] = {'gold':g, 'pred':p, 'correct':tp,
                          'precision':precision, 'recall':recall, 'f1':f1}

    total_g, total_p, total_tp = sum(gold_count.values()), sum(pred_count.values()), sum(right_count.values())
    macro_f1 = sum(x['f1'] for x in per_class.values()) / len(LABELS)
    print(f'\n{name}: {len(doc_ids)} docs | Gold {total_g} | Pred {total_p} | Exact {total_tp} | Macro F1 {macro_f1:.4f}')
    print('Label          Gold  Pred  Exact    P      R      F1')
    for lab, row in per_class.items():
        print(f"{lab:13} {row['gold']:5} {row['pred']:5} {row['correct']:5} "
              f"{row['precision']:.3f}  {row['recall']:.3f}  {row['f1']:.3f}")
    print('Missed examples:', [(lab, texts[did][a:b]) for did, (a,b,lab) in missed[:5]])
    print('Extra examples:', [(lab, texts[did][a:b]) for did, (a,b,lab) in extra[:5]])
    return {'macro_f1':macro_f1, 'gold':total_g, 'pred':total_p,
            'exact':total_tp, 'per_label':per_class}

# One held-out split and a sample of documents actually used for training.
val_report = span_report('Validation', dev_ids)
train_report = span_report('Train sample', train_ids[:20])

# BIO consistency bhi check karo: I-X se pehle B-X / I-X hona chahiye.
@torch.no_grad()
def bio_report(dataset, n_windows=100):
    was_training = model.training
    model.eval()
    pred_counts, gold_counts, invalid = Counter(), Counter(), Counter()
    try:
        step = max(1, len(dataset) // n_windows)
        for i in list(range(0, len(dataset), step))[:n_windows]:
            row = dataset[i]     # Humare Dataset keys: ids, offsets, tags
            x = torch.tensor([row['ids']], dtype=torch.long, device=DEVICE)
            pred_ids = model(x)[0].argmax(-1).cpu().tolist()
            prev = 'O'
            for p, g in zip(pred_ids, row['tags']):
                predicted, actual = TAGS[p], TAGS[g]
                pred_counts[predicted] += 1
                gold_counts[actual] += 1
                if predicted.startswith('I-'):
                    lab = predicted[2:]
                    if prev not in ('B-'+lab, 'I-'+lab): invalid[lab] += 1
                prev = predicted
    finally:
        model.train(was_training)
    total = sum(pred_counts.values())
    inside = sum(v for k,v in pred_counts.items() if k.startswith('I-'))
    print('BIO tokens checked:',total)
    print('Predicted O:',round(100*pred_counts['O']/max(1,total),2),'%')
    print('Gold O:',round(100*gold_counts['O']/max(1,total),2),'%')
    print('Invalid I transitions:',sum(invalid.values()),'of',inside)
    print('Most invalid classes:',invalid.most_common(8))
    return {'predicted':dict(pred_counts),'gold':dict(gold_counts),'invalid_I':dict(invalid)}

bio_stats = bio_report(train_ds)
with open(OUTPUT/'diagnostics.json','w') as f:
    json.dump({'validation':val_report,'training':train_report,'bio':bio_stats},f,indent=2)



Validation: 7 docs | Gold 721 | Pred 868 | Exact 421 | Macro F1 0.3535
Label          Gold  Pred  Exact    P      R      F1
APP              17    33     2 0.061  0.118  0.080
RESP             64    11     4 0.364  0.062  0.107
A.COUNSEL        16    22     5 0.227  0.312  0.263
R.COUNSEL        23    27    13 0.481  0.565  0.520
JUDGE            42    35     8 0.229  0.190  0.208
WIT               0    10     0 0.000  0.000  0.000
AUTH             27    89    19 0.213  0.704  0.328
COURT           181   225   147 0.653  0.812  0.724
STAT             52    68    23 0.338  0.442  0.383
PREC             81   104    25 0.240  0.309  0.270
DATE            155   174   142 0.816  0.916  0.863
CASENO           63    70    33 0.471  0.524  0.496
Missed examples: [('APP', 'SREE SREE ISHWAR SRIDHAR JEW'), ('APP', 'Kashinath Dutt'), ('CASENO', 'suit No. 969 of (1) (1920) 24 C.W.N. 1926'), ('APP', 'Pashupati Nath Dutt'), ('APP', 'Sridhar Jew Thakur')]
Extra examples: [('PREC', 'Bhupendra'), ('WIT

In [12]:

# Reload the selected BEST Phase-2 checkpoint, then predict official fold3.
# Exact Codabench output: {doc_id: [[start_char,end_char,entity_label], ...]}.
checkpoint=torch.load(CKPT,map_location='cpu',weights_only=True)
assert checkpoint['labels']==TAGS and checkpoint['technique']==TECHNIQUE
model.load_state_dict(checkpoint['model_state'],strict=True)
model.eval()

# Same trained weights, just compare greedy vs valid BIO decoding on validation.
legacy_metrics = strict_scores(dev_ids, method='legacy')
greedy_metrics = strict_scores(dev_ids, method='greedy')
viterbi_metrics = strict_scores(dev_ids, method='viterbi')
print('Validation F1 - old window method:', round(legacy_metrics['strict_macro_f1'], 4))
print('Validation F1 - stitched greedy:', round(greedy_metrics['strict_macro_f1'], 4))
print('Validation F1 - stitched Viterbi:', round(viterbi_metrics['strict_macro_f1'], 4))
decoder_f1 = {
    'legacy': legacy_metrics['strict_macro_f1'],
    'greedy': greedy_metrics['strict_macro_f1'],
    'viterbi': viterbi_metrics['strict_macro_f1'],
}
if AUTO_SELECT_DECODER:
    # Test ke labels unknown hain, isliye method sirf validation se choose karte hain.
    DECODE_METHOD = max(decoder_f1, key=decoder_f1.get)
print('Submission decoding:', DECODE_METHOD)
with open(OUTPUT/'decoder_comparison.json', 'w') as f:
    json.dump({'scores': decoder_f1, 'selected': DECODE_METHOD}, f, indent=2)

test_texts=extract_documents(read_official_json('Test/fold3-text.json'))
assert len(test_texts)>0
predictions={}
for i,(docid,doc_text) in enumerate(test_texts.items()):
    predictions[docid]=[[int(a),int(b),lab] for a,b,lab in predict_one_doc(doc_text)]
    if (i+1)%20==0:print('Predicted',i+1,'/',len(test_texts))

assert set(predictions)==set(test_texts)
for did,entities in predictions.items():
    for item in entities:
        assert len(item)==3 and isinstance(item[0],int) and isinstance(item[1],int)
        assert 0<=item[0]<item[1]<=len(test_texts[did]) and item[2] in LABELS

pred_path=OUTPUT/'prediction.json'
with open(pred_path,'w',encoding='utf-8') as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)
zip_path=OUTPUT/'submission.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path,arcname='prediction.json')
with zipfile.ZipFile(zip_path) as zf:
    assert zf.namelist()==['prediction.json']
    verified=json.loads(zf.read('prediction.json'))
    assert set(verified)==set(test_texts)
print('Ready for Codabench:',zip_path)
print('Total test documents:',len(predictions),'predicted spans:',sum(map(len,predictions.values())))


Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Validation F1 - old window method: 0.2362
Validation F1 - stitched greedy: 0.2386
Validation F1 - stitched Viterbi: 0.3535
Submission decoding: viterbi


fold3-text.json: 0.00B [00:00, ?B/s]

Predicted 20 / 35
Ready for Codabench: /kaggle/working/phase2_a1_adapters_v2_30ep/submission.zip
Total test documents: 35 predicted spans: 4627


In [13]:
# Run AFTER prediction.json and submission.zip are created.
# Ye cell Kaggle ke experiment folder ki saari files HF par save karta hai.
from pathlib import Path
from huggingface_hub import HfApi

UPLOAD_TO_HF = True  # Agar upload nahi karna toh False rakho
PHASE2_REPO_NAME = 'cs781-phase2-a1_adapters-v2'

if UPLOAD_TO_HF:
    # Pehle check karo ki predictions ban chuke hain, half-finished run upload nahi karna.
    must_have = [CKPT, OUTPUT / 'prediction.json', OUTPUT / 'submission.zip']
    missing = [str(p) for p in must_have if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError('Final output missing: ' + ', '.join(missing))

    # Hugging Face repo private rakha hai, outputs accidentally public nahi honge.
    hf_api = HfApi(token=HF_TOKEN)
    hf_user = hf_api.whoami()['name']
    repo_id = hf_user + '/' + PHASE2_REPO_NAME
    hf_api.create_repo(repo_id=repo_id, repo_type='model', private=True, exist_ok=True)
    if not hf_api.repo_info(repo_id=repo_id, repo_type='model').private:
        raise ValueError('HF repo is public. Change its visibility to private before uploading.')

    # Pura experiment folder upload hoga: model, history, diagnostics, JSON aur zip.
    hf_api.upload_folder(folder_path=str(OUTPUT), repo_id=repo_id,
                         repo_type='model', commit_message='Save final Phase 2 outputs')

    # Phase 1 tokenizer/config bhi saath rakh rahe hain, inference mein kaam aayenge.
    for name in ('architecture_config.json', 'tokenizer.json',
                 'tokenizer_config.json', 'special_tokens_map.json'):
        source_file = P1_DIR / name
        if source_file.is_file():
            hf_api.upload_file(path_or_fileobj=str(source_file),
                               path_in_repo=name, repo_id=repo_id, repo_type='model')

    print('Uploaded experiment files to:', 'https://huggingface.co/' + repo_id)
    print('Files from:', OUTPUT)
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file():
            print(' -', path.relative_to(OUTPUT))
else:
    print('HF upload skipped; files are still in:', OUTPUT)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded experiment files to: https://huggingface.co/P-261110044/cs781-phase2-a1_adapters-v2
Files from: /kaggle/working/phase2_a1_adapters_v2_30ep
 - a1_adapters_v2_30ep_best.pt
 - decoder_comparison.json
 - diagnostics.json
 - history.json
 - phase1_files/model.safetensors
 - phase1_files/tokenizer.json
 - phase1_files/tokenizer_config.json
 - phase1_used.json
 - prediction.json
 - submission.zip
